# NASA POWER – Hourly Iowa 2012–2025 Dataset

Fetches hourly **temperature** (T2M), **precipitation** (PRECTOTCORR), and **wind speed** (WS10M) from the NASA POWER API for all of Iowa, 2012–2025, using the 96 native-resolution grid cells (0.5° lat × 0.625° lon, ~56×52 km each).

### Iowa bounding box
| | Value |
|---|---|
| Latitude | 40.37 – 43.50 N → 8 rows spaced 0.5° |
| Longitude | -96.64 – -90.14 W → 12 cols spaced 0.625° |
| Total cells | 96 |

### Output
- `iowa_hourly_2012_2025.csv` (~700 MB, ~11.8 million rows)
- Columns: `lat`, `lon`, `datetime_utc`, `T2M_C`, `PRECTOTCORR_mm_hr`, `WS10M_m_s`

### Notes
- Only **96 API requests** are made (one per unique cell, full 14-year range).
- Runtime is typically **2–5 minutes** depending on server response time.
- Progress is saved after each cell so you can **resume if interrupted** — re-run from the fetch loop cell and already-fetched cells will be skipped automatically.

## 1. Install dependencies

In [ ]:
%pip install requests pandas tqdm numpy --quiet

## 2. Imports

In [ ]:
import os
import time
import requests
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

## 3. Configuration

Edit the values below to change the geographic extent, date range, parameters, or output paths.

In [ ]:
# ── Iowa bounding box ─────────────────────────────────────────────────────────
LAT_MIN, LAT_MAX = 40.37, 43.50
LON_MIN, LON_MAX = -96.64, -90.14

# Native POWER resolution
LAT_STEP = 0.5
LON_STEP = 0.625

# ── NASA POWER API settings ───────────────────────────────────────────────────
BASE_URL   = "https://power.larc.nasa.gov/api/temporal/hourly/point"
COMMUNITY  = "AG"
PARAMETERS = "T2M,PRECTOTCORR,WS10M"
START      = "20120101"
END        = "20251231"
FORMAT     = "JSON"
TIME_STD   = "UTC"

HEADERS       = {"User-Agent": "Iowa-Climate-Research/1.0"}
RETRY_WAIT    = 10   # seconds between retries
MAX_RETRIES   = 3
REQUEST_DELAY = 0.5  # seconds between cells

# ── Output paths ──────────────────────────────────────────────────────────────
OUT_FINAL   = "iowa_hourly_2012_2025.csv"
OUT_PARTIAL = "iowa_hourly_2012_2025_partial.csv"
DONE_LOG    = "iowa_hourly_2012_2025_done.txt"

## 4. Build the native-resolution grid

In [ ]:
lats = np.arange(LAT_MIN, LAT_MAX + LAT_STEP, LAT_STEP)
lons = np.arange(LON_MIN, LON_MAX + LON_STEP, LON_STEP)
grid_points = [(round(float(lat), 4), round(float(lon), 4))
               for lat in lats for lon in lons]

print(f"Native-resolution grid: {len(lats)} lat rows × {len(lons)} lon cols "
      f"= {len(grid_points)} unique cells")
print(f"Date range : {START} → {END}  ({int(END[:4]) - int(START[:4]) + 1} years)")
print(f"Parameters : {PARAMETERS}")

## 5. Load resume state

If the fetch was previously interrupted, already-completed cells are read from the done log and skipped.

In [ ]:
done_cells = set()

if os.path.exists(DONE_LOG):
    with open(DONE_LOG) as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) == 2:
                done_cells.add((float(parts[0]), float(parts[1])))
    print(f"Resuming – {len(done_cells)} cells already fetched, "
          f"{len(grid_points) - len(done_cells)} remaining.")
else:
    print("No previous progress found — starting fresh.")

## 6. Define the fetch function

In [ ]:
def fetch_point(lat: float, lon: float) -> pd.DataFrame | None:
    """Fetch the full hourly time series for one grid cell."""
    params = {
        "parameters":    PARAMETERS,
        "community":     COMMUNITY,
        "longitude":     lon,
        "latitude":      lat,
        "start":         START,
        "end":           END,
        "format":        FORMAT,
        "time-standard": TIME_STD,
    }

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(BASE_URL, params=params,
                                headers=HEADERS, timeout=180)
            if resp.status_code == 200:
                data  = resp.json()
                props = data.get("properties", {}).get("parameter", {})
                if not props:
                    print(f"  ⚠ Empty response for ({lat}, {lon})")
                    return None

                timestamps = list(next(iter(props.values())).keys())
                df = pd.DataFrame({"timestamp": timestamps})
                df["lat"] = lat
                df["lon"] = lon

                for param, values in props.items():
                    df[param] = [values.get(ts, float("nan")) for ts in timestamps]

                df["datetime_utc"] = pd.to_datetime(df["timestamp"], format="%Y%m%d%H")
                df = df.drop(columns=["timestamp"])
                df = df.rename(columns={
                    "T2M":         "T2M_C",
                    "PRECTOTCORR": "PRECTOTCORR_mm_hr",
                    "WS10M":       "WS10M_m_s",
                })

                # Replace NASA POWER fill value
                for col in ["T2M_C", "PRECTOTCORR_mm_hr", "WS10M_m_s"]:
                    df[col] = df[col].replace(-999, float("nan"))

                return df[["lat", "lon", "datetime_utc",
                            "T2M_C", "PRECTOTCORR_mm_hr", "WS10M_m_s"]]

            elif resp.status_code == 429:
                wait = RETRY_WAIT * attempt
                print(f"  Rate limited – sleeping {wait}s …")
                time.sleep(wait)
            else:
                print(f"  HTTP {resp.status_code} for ({lat}, {lon}): "
                      f"{resp.text[:200]}")
                return None

        except Exception as exc:
            print(f"  Attempt {attempt} failed for ({lat}, {lon}): {exc}")
            time.sleep(RETRY_WAIT)

    print(f"  ✗ Giving up on ({lat}, {lon}) after {MAX_RETRIES} attempts")
    return None

## 7. Fetch all grid cells

Progress is appended to `iowa_hourly_2012_2025_partial.csv` after each successful cell. Re-run this cell to resume after an interruption.

In [ ]:
failed = []
first_write = not os.path.exists(OUT_PARTIAL)
remaining = [pt for pt in grid_points if pt not in done_cells]

print(f"Fetching {len(remaining)} remaining cells …\n")

for lat, lon in tqdm(remaining, desc="Fetching cells", unit="cell"):
    df = fetch_point(lat, lon)

    if df is not None:
        df.to_csv(OUT_PARTIAL, mode="a", header=first_write, index=False)
        first_write = False

        with open(DONE_LOG, "a") as f:
            f.write(f"{lat},{lon}\n")
    else:
        failed.append((lat, lon))

    time.sleep(REQUEST_DELAY)

print(f"\nFetch complete. Failed cells: {failed if failed else 'none'}")

## 8. Finalise output

Renames the partial file to the final output and removes the done log.

In [ ]:
if os.path.exists(OUT_PARTIAL):
    os.rename(OUT_PARTIAL, OUT_FINAL)
    if os.path.exists(DONE_LOG):
        os.remove(DONE_LOG)
    print(f"✅ Partial file renamed to: {OUT_FINAL}")
elif os.path.exists(OUT_FINAL):
    print(f"✅ Final file already exists: {OUT_FINAL}")
else:
    print("⚠  No data was retrieved. Check your internet connection.")

## 9. Summary & preview

In [ ]:
if os.path.exists(OUT_FINAL):
    row_count = sum(1 for _ in open(OUT_FINAL)) - 1  # subtract header
    cells_fetched = len(grid_points) - len(failed)

    print(f"Output file  : {OUT_FINAL}")
    print(f"Total rows   : {row_count:,}")
    print(f"Cells fetched: {cells_fetched}/{len(grid_points)}")
    if failed:
        print(f"Failed cells : {failed}")

    sample = pd.read_csv(OUT_FINAL, nrows=5)
    display(sample)
else:
    print("⚠  Output file not found.")

## 10. Quick exploratory stats (optional)

In [ ]:
if os.path.exists(OUT_FINAL):
    # Read a larger sample for stats (adjust nrows as needed)
    df_sample = pd.read_csv(OUT_FINAL, nrows=100_000, parse_dates=["datetime_utc"])
    display(df_sample.describe())